# inference

> Turn a trained model into predictions: probability maps (optionally test-time augmented), label decoding, and an operating-threshold search for binary tasks.

`predict` returns per-class probabilities for both binary (sigmoid) and multi-class (softmax). `decode` collapses them to a label mask. `find_operating_threshold` sweeps the binary decision threshold to hit a target recall, which matters when a miss is far costlier than a false alarm.

In [1]:
#| default_exp inference

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import cv2

from typing import Dict, List, Optional, Sequence

import numpy as np
import torch

In [4]:
#| exporti
from segmentation_tools_pt.metrics import ObjectMetrics

## Predict probabilities

One forward pass returns `(C, H, W)` probabilities. With `tta=True` we average predictions over the four 90-degree rotations, rotating each prediction back before averaging. TTA trades compute for a small, usually-positive accuracy bump on thin structures.

`task` disambiguates `C > 1`: `multiclass` (default) uses softmax -- classes are mutually exclusive, so probabilities sum to 1 across channels. `multilabel` uses an independent sigmoid per channel instead, so overlapping classes (e.g. `void` inside `chip`) aren't forced to compete for probability mass. `binary` (`C == 1`) always uses sigmoid regardless of `task`.

### Shared activation rule

Both `predict` and `predict_tiled` must turn logits into probabilities the same way: **sigmoid** when the head is binary (`C == 1`) or `task='multilabel'` (channels independent, overlapping classes allowed), **softmax** across channels for `multiclass` (mutually exclusive classes), and a pure **passthrough** when `activate=False`. That rule used to live twice -- once inside `predict`'s `forward`, once inside `predict_tiled`'s `activate_probs` -- so the two copies could drift and the same model would get different probabilities depending on image size (issue #25).

Plan (experiment -> export -> test):

1. Experiment: run the rule inline on synthetic logits for the four pinned cases (binary sigmoid, multilabel sigmoid, multiclass softmax, logits passthrough).
2. Promote it to `_activate(logits, task, activate)`; both public functions delegate to it. The leading underscore keeps it private -- out of `__all__` -- so callers keep using `predict` / `predict_tiled`.
3. Test the four cases against the export, on the same synthetic logits.

In [ ]:
# experiment: the activation rule, inline, on synthetic logits — not exported yet
logits_1c = torch.tensor([[[-2.0, 0.0, 3.0]]])          # (1, 1, 3) binary head
logits_3c = torch.tensor([[[ 2.0, 0.0, -1.0],
                           [ 0.0, 1.0,  2.0],
                           [ 1.0, 1.0,  1.0]]])          # (1, 3, 3) 3-class head

def _activate_proto(logits, task, activate):
    if not activate: return logits
    return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, dim=1)

cases = [
    ('binary',      _activate_proto(logits_1c, None, True)),
    ('multilabel',  _activate_proto(logits_3c, 'multilabel', True)),
    ('multiclass',  _activate_proto(logits_3c, None, True)),
    ('passthrough', _activate_proto(logits_3c, None, False)),
]
for name, a in cases:
    print(f'{name:12s} shape={tuple(a.shape)} range=[{a.min():+.3f}, {a.max():+.3f}]')

# verify each branch
assert torch.equal(_activate_proto(logits_1c, None, True), torch.sigmoid(logits_1c))
assert torch.equal(_activate_proto(logits_1c, 'multiclass', True), torch.sigmoid(logits_1c))  # C==1 wins over task
assert torch.equal(_activate_proto(logits_3c, 'multilabel', True), torch.sigmoid(logits_3c))
mc = _activate_proto(logits_3c, None, True)
assert torch.allclose(mc.sum(1), torch.ones(1, 3))  # softmax sums to 1 across channels
assert torch.equal(_activate_proto(logits_3c, None, False), logits_3c)
print('activation rule verified for all four cases')

In [ ]:
#| export
def _activate(
    logits: torch.Tensor,  # (B, C, H, W) raw model output
    task: str,             # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
    activate: bool,        # if False, return logits untouched
) -> torch.Tensor:
    """Sigmoid if `C==1` or `task='multilabel'`, softmax for multiclass, logits passthrough if `activate=False`.

    The one activation rule shared by `predict` and `predict_tiled` (issue #25), kept
    private so the two public entry points can never drift apart.

    Example:
        >>> _activate(torch.tensor([[[-2.0, 3.0]]]), None, True).round(decimals=3).tolist()
        [[[0.119, 0.953]]]
    """
    if not activate: return logits
    return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, dim=1)

In [ ]:
#| test
# test: the four pinned activation cases (issue #25) — one rule, two callers
assert torch.equal(_activate(logits_1c, None, True), torch.sigmoid(logits_1c))            # binary C==1 -> sigmoid
assert torch.equal(_activate(logits_1c, 'multiclass', True), torch.sigmoid(logits_1c))    # C==1 wins over task
assert torch.equal(_activate(logits_3c, 'multilabel', True), torch.sigmoid(logits_3c))    # multilabel -> per-channel sigmoid
mc = _activate(logits_3c, None, True)
assert torch.allclose(mc.sum(1), torch.ones(1, 3))                                        # multiclass -> softmax over channels
assert torch.equal(_activate(logits_3c, None, False), logits_3c)                          # activate=False -> logits untouched
print('_activate ✓  sigmoid / softmax / passthrough pinned')

In [5]:
#| export
@torch.no_grad()
def predict(
    model,                 # trained model returning (B, C, H, W) logits
    image: torch.Tensor,   # a single (C, H, W) input
    device: torch.device,  # cpu or cuda
    tta: bool = False,     # average over 90-degree rotations
    n_augs: int = 4,       # number of rotations when tta=True
    activate: bool = True, # apply sigmoid/softmax to logits
    task: str = None,      # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities: sigmoid if `C==1` or `task='multilabel'`, else softmax."""
    model.eval()
    x = image.unsqueeze(0).to(device)

    def forward(t: torch.Tensor) -> torch.Tensor:
        return _activate(model(t), task, activate)

    if not tta:
        return forward(x).squeeze(0)
    preds = []
    for k in range(n_augs):
        pr = forward(torch.rot90(x, k, dims=[2, 3]))
        preds.append(torch.rot90(pr, -k, dims=[2, 3]))
    return torch.stack(preds).mean(0).squeeze(0)

In [6]:
#| export
def decode(
    probs: torch.Tensor,     # (C, H, W) probabilities
    threshold=0.5,            # decision threshold: float (binary/multilabel-shared), or per-channel Sequence[float] for multilabel
    task: str = None,         # 'binary' | 'multiclass' | 'multilabel'; None infers binary (C==1) or multiclass (C>1)
) -> torch.Tensor:
    """Collapse probabilities to a label mask.

    `binary`/`multiclass` (default behavior, unchanged) return `(H, W)`: a
    thresholded `{0,1}` mask for `C==1`, or an argmax label map for `C>1`.
    `multilabel` returns `(C, H, W)` -- classes can overlap, so there is no
    single `(H,W)` label map to collapse to; each channel is thresholded
    independently instead.

    Example:
        >>> decode(torch.tensor([[[0.2, 0.9]]])).tolist()
        [[0, 1]]
    """
    if task == 'multilabel':
        c = probs.shape[0]
        # duck-type "sequence of per-channel thresholds" rather than isinstance(list, tuple):
        # Hydra/OmegaConf configs hand back ListConfig, not a plain list, for a YAML list.
        is_seq = hasattr(threshold, '__len__') and not isinstance(threshold, (str, bytes))
        t = list(threshold) if is_seq else [threshold] * c
        return torch.stack([probs[k] >= t[k] for k in range(c)]).long()
    if probs.shape[0] == 1:
        return (probs[0] >= threshold).long()
    return probs.argmax(0)

In [7]:
#| test
# test: shapes for binary and multi-class, with and without TTA
import torch.nn as nn
dev = torch.device('cpu')
mb = nn.Conv2d(3, 1, 3, padding=1)
p = predict(mb, torch.rand(3, 16, 16), dev)
assert p.shape == (1, 16, 16)
assert predict(mb, torch.rand(3, 16, 16), dev, tta=True).shape == (1, 16, 16)
assert decode(p).shape == (16, 16)
mc = nn.Conv2d(3, 3, 3, padding=1)
pc = predict(mc, torch.rand(3, 16, 16), dev)
assert pc.shape == (3, 16, 16) and decode(pc).shape == (16, 16)

In [8]:
#| test
# test: multilabel predict/decode -- independent per-channel sigmoid, no softmax, overlap preserved
m3 = nn.Conv2d(3, 3, 3, padding=1)
p3 = predict(m3, torch.rand(3, 16, 16), dev, task='multilabel')
assert p3.shape == (3, 16, 16)
assert not torch.allclose(p3.sum(0), torch.ones(16, 16))  # not softmax-normalised

d3 = decode(p3, task='multilabel')
assert d3.shape == (3, 16, 16)  # unlike multiclass decode, NOT collapsed to (H,W)

# per-channel thresholds actually take effect
probs_fixed = torch.full((3, 4, 4), 0.6)
d_low = decode(probs_fixed, threshold=0.5, task='multilabel')
d_high = decode(probs_fixed, threshold=[0.5, 0.7, 0.5], task='multilabel')
assert bool(d_low[1].all()) and not bool(d_high[1].any())  # channel 1: 0.6>=0.5 fires, 0.6>=0.7 doesn't
print('multilabel predict/decode ✓  independent sigmoid, (C,H,W) decode, per-channel thresholds')

multilabel predict/decode ✓  independent sigmoid, (C,H,W) decode, per-channel thresholds


## Hierarchy post-process (multilabel)

`decode(task='multilabel')` thresholds each channel independently, so a
trained model can still predict a `void`/`lifted` pixel outside its `chip`
footprint even with `losses.hierarchy_regularizer_loss` nudging it during
training (that's a soft penalty, not a guarantee). `apply_hierarchy` hard-
enforces the containment relationship at inference time: any sub-channel
pixel not also covered by its super-channel is zeroed out.

In [9]:
#| export
def apply_hierarchy(
    masks: torch.Tensor,  # (C, H, W) or (B, C, H, W) decoded/binary multilabel masks
    pairs,                  # [(sub_idx, super_idx), ...], e.g. [(1, 0), (2, 0)] for (void,chip),(lifted,chip)
) -> torch.Tensor:
    """Zero out sub-channel pixels not covered by their super-channel. Returns a new tensor."""
    if not pairs:
        return masks
    masks = masks.clone()
    c_dim = -3  # channel is always third-from-last: (...,C,H,W)
    for s, p in pairs:
        sub = masks.select(c_dim, s)
        sup = masks.select(c_dim, p)
        if masks.dtype == torch.bool:
            sub &= sup
        else:
            sub.mul_(sup)
    return masks

In [10]:
#| test
# test: apply_hierarchy zeroes sub-channel pixels outside their super-channel, leaves the rest alone
chip = torch.zeros(4, 4); chip[1:3, 1:3] = 1
void = torch.zeros(4, 4); void[0:2, 0:2] = 1  # only (1,1) is inside chip; (0,0),(0,1),(1,0) leak outside
masks = torch.stack([chip, void]).long()  # (C=2, H, W): 0=chip, 1=void

fixed = apply_hierarchy(masks, pairs=[(1, 0)])
assert fixed[1, 1, 1] == 1        # inside chip -> kept
assert fixed[1, 0, 0] == 0        # outside chip -> zeroed
assert torch.equal(fixed[0], masks[0])  # super-channel (chip) itself is untouched
assert masks[1, 0, 0] == 1        # original input not mutated (clone, not in-place on the caller's tensor)

# batched (B, C, H, W) works the same way
batched = masks.unsqueeze(0).repeat(2, 1, 1, 1)
fixed_b = apply_hierarchy(batched, pairs=[(1, 0)])
assert fixed_b.shape == (2, 2, 4, 4)
assert bool((fixed_b[:, 1, 0, 0] == 0).all())

# empty pairs -> no-op, same tensor object back (cheap default)
assert apply_hierarchy(masks, []) is masks

# also works on a bool mask (as produced by decode(..., task='multilabel'))
bool_masks = masks.bool()
fixed_bool = apply_hierarchy(bool_masks, [(1, 0)])
assert fixed_bool.dtype == torch.bool and bool(fixed_bool[1, 0, 0]) is False
print('apply_hierarchy ✓  hard-enforces containment, (C,H,W) + (B,C,H,W) + bool, no input mutation')

apply_hierarchy ✓  hard-enforces containment, (C,H,W) + (B,C,H,W) + bool, no input mutation


## Class-targeted morph post-process (multilabel)

A per-channel sigmoid decoder can leave tiny speckle false alarms in one class
channel (e.g. `void`) that nothing during training punishes — deliberately so:
the loss and the raw train metrics must judge the model as-is. The cleanup
therefore belongs strictly to the inference/production decode step, and it is
opt-in config (`postprocess.class_morph.*`, master switch default **false**,
resolved by `hydra_pipeline.resolve_class_morph_cfg`).

`apply_class_morph` cleans **one named class channel** with classical
morphology: an opening (erode -> dilate, kills thin speckles) followed by a
connected-component min-area filter (kills small blobs that survived the
opening). All other channels pass through untouched. Fixed decode order:

> threshold decode (`decode`) -> class morph (`apply_class_morph`) -> hierarchy (`apply_hierarchy`)

Plan:
1. Experiment: a decoded 3-channel mask with 1-px speckles in one channel — verify opening + min-area clean only that channel.
2. Export `apply_class_morph`: shape-generic `(C,H,W)`/`(B,C,H,W)`, no input mutation, clear error on bad `class_idx`.
3. Test the acceptance criteria, including the fixed morph-then-hierarchy order.

In [11]:
# experiment: a decoded (C,H,W) mask with 1-px speckles in the void channel only
import cv2
import numpy as np

chip = torch.ones(12, 12).long()
void = torch.zeros(12, 12).long()
void[4:8, 4:8] = 1      # real 16-px blob
void[0, 0] = 1          # 1-px speckle
void[11, 11] = 1        # 1-px speckle
lifted = torch.zeros(12, 12).long(); lifted[4:8, 4:8] = 1

# inline logic first: opening + min-area on the void channel by hand
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
m = (void.numpy() > 0).astype(np.uint8)
m = cv2.erode(m, kernel, borderValue=0)   # outside-image = background
m = cv2.dilate(m, kernel, borderValue=0)
n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
for lbl in range(1, n):
    if stats[lbl, cv2.CC_STAT_AREA] < 8:
        m[labels == lbl] = 0
print(f'void fg: before={int(void.sum())} after={int(m.sum())}')  # expect 18 -> 16
assert int(m.sum()) == 16 and m[4:8, 4:8].all()  # speckles gone, blob intact

void fg: before=18 after=16


In [12]:
#| export
def apply_class_morph(
    masks: torch.Tensor,   # (C, H, W) or (B, C, H, W) decoded/binary multilabel masks
    class_idx: int,        # channel index of the class to clean (resolve class_name via class_names first)
    kernel_size: int = 3,  # opening structuring element side (odd); kills thin speckles
    min_area: int = 16,    # drop connected components smaller than this many pixels (1 = keep all)
) -> torch.Tensor:
    """Morph-clean ONE class channel (opening + min-area); other channels pass through untouched.

    Inference/production decode only — never applied to the training loss or to
    raw train metrics (those must judge the model as-is). Fixed decode order:
    `decode` -> `apply_class_morph` -> `apply_hierarchy`.

    Example:
        >>> cleaned = apply_class_morph(decoded, class_idx=1, kernel_size=3, min_area=16)
    """
    c_dim = -3  # channel is always third-from-last: (..., C, H, W)
    n_classes = masks.shape[c_dim]
    if not 0 <= class_idx < n_classes:
        raise ValueError(
            f'class_idx={class_idx} is out of range for a {n_classes}-channel mask. '
            'Resolve postprocess.class_morph.class_name against data.class_names first '
            '(see hydra_pipeline.resolve_class_morph_cfg).'
        )
    if kernel_size < 1:
        raise ValueError(f'kernel_size must be >= 1, got {kernel_size}')
    if min_area < 1:
        raise ValueError(f'min_area must be >= 1, got {min_area}')
    masks = masks.clone()
    ch = masks.select(c_dim, class_idx)  # view into the clone; writes below land in the clone
    items = ch if ch.dim() > 2 else ch.unsqueeze(0)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    for i, item in enumerate(items):
        m = (item.numpy() > 0).astype(np.uint8)
        # opening as explicit erode->dilate with outside-image = background:
        # border-touching speckles are removable, at the cost of real objects
        # touching the border losing up to kernel_size // 2 px at the boundary
        m = cv2.erode(m, kernel, borderValue=0)
        m = cv2.dilate(m, kernel, borderValue=0)
        if min_area > 1:
            n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
            for lbl in range(1, n):
                if stats[lbl, cv2.CC_STAT_AREA] < min_area:
                    m[labels == lbl] = 0
        items[i] = torch.from_numpy(m.astype(bool)).to(masks.dtype)
    return masks

In [13]:
#| test
# test: issue #19 acceptance — named channel cleaned, others untouched, no mutation, clear errors
chip = torch.ones(12, 12).long()
void = torch.zeros(12, 12).long(); void[4:8, 4:8] = 1; void[0, 0] = 1; void[11, 11] = 1
lifted = torch.zeros(12, 12).long(); lifted[4:8, 4:8] = 1
masks = torch.stack([chip, void, lifted])  # (C=3, H, W): 0=chip, 1=void, 2=lifted

cleaned = apply_class_morph(masks, class_idx=1, kernel_size=3, min_area=8)
assert int(cleaned[1].sum()) == 16 and bool(cleaned[1][4:8, 4:8].all())  # speckles gone, blob kept
assert torch.equal(cleaned[0], masks[0]) and torch.equal(cleaned[2], masks[2])  # others untouched
assert int(masks[1].sum()) == 18  # caller's tensor not mutated

# batched (B, C, H, W) behaves identically per batch item
batched = masks.unsqueeze(0).repeat(2, 1, 1, 1)
cleaned_b = apply_class_morph(batched, class_idx=1, kernel_size=3, min_area=8)
assert cleaned_b.shape == (2, 3, 12, 12)
assert int(cleaned_b[0, 1].sum()) == 16 and int(cleaned_b[1, 1].sum()) == 16

def _raises(fn, msg):
    try:
        fn()
        raise AssertionError(f'expected ValueError: {msg}')
    except ValueError as e:
        assert msg in str(e), str(e)

_raises(lambda: apply_class_morph(masks, class_idx=5), 'class_idx=5 is out of range')
_raises(lambda: apply_class_morph(masks, class_idx=1, kernel_size=0), 'kernel_size must be >= 1')
_raises(lambda: apply_class_morph(masks, class_idx=1, min_area=0), 'min_area must be >= 1')

# fixed order: threshold decode -> class morph (named class) -> hierarchy.
# Morph runs on the SUPER channel (chip): its small speckle dies, and hierarchy
# then drops the void pixels that lived under it. Reversed order would strand
# those void pixels inside the (no longer existing) chip speckle.
probs = torch.full((3, 12, 12), 0.1)
probs[0, 2:10, 2:10] = 0.9  # chip main blob
probs[0, 0:2, 0:2] = 0.9    # separate small chip speckle (decodes on, opening kills it)
probs[1, 0:2, 0:2] = 0.9    # void only inside that chip speckle (subset, no leak)
probs[2, 4:8, 4:8] = 0.9    # lifted inside the chip main blob
decoded = decode(probs, threshold=0.5, task='multilabel')
morphed = apply_class_morph(decoded, class_idx=0, kernel_size=3, min_area=8)
final = apply_hierarchy(morphed, pairs=[(1, 0), (2, 0)])
assert not bool(final[0, 0:2, 0:2].any())  # chip speckle removed by morph
assert bool(final[0, 2:10, 2:10].all())    # chip main blob survives opening
assert not bool(final[1].any())            # void died with its super: morph ran BEFORE hierarchy
assert bool(final[2, 4:8, 4:8].all())      # untouched channel survives the full chain
# master switch off = skip apply_class_morph entirely: nothing cleans the speckle
no_morph = apply_hierarchy(decoded, pairs=[(1, 0), (2, 0)])
assert bool(no_morph[1, 0:2, 0:2].all())  # void speckle survives under the chip speckle
print('apply_class_morph ✓  named channel cleaned, others untouched, batched, errors, morph→hierarchy order')

apply_class_morph ✓  named channel cleaned, others untouched, batched, errors, morph→hierarchy order


## Operating-threshold search (binary)

Sweep the sigmoid threshold over a validation loader and report pixel recall/precision/IoU plus object-level detection at each. We recommend the **highest** threshold that still meets `target_recall`: highest precision among the options that do not start missing targets.

In [14]:
#| export
@torch.no_grad()
def find_operating_threshold(
    model,                          # trained binary model
    val_loader,                     # validation DataLoader of {'image','mask'}
    device: torch.device,           # cpu or cuda
    target_recall: float = 0.99,    # minimum pixel recall to accept
    overlap_thresh: float = 0.5,    # object overlap fraction for detection
    thresholds: Optional[Sequence[float]] = None,  # thresholds to scan
) -> Dict:
    """Scan thresholds; return the recommended one plus the full sweep."""
    model.eval()
    thresholds = list(thresholds) if thresholds is not None else list(np.arange(0.05, 0.95, 0.05))
    prob_imgs, gt_imgs = [], []
    for batch in val_loader:
        images = batch['image'].to(device)
        p = torch.sigmoid(model(images))[:, 0].cpu().numpy()
        prob_imgs.append(p)
        gt_imgs.append((batch['mask'].numpy() > 0).astype(np.uint8))
    probs = np.concatenate(prob_imgs)        # (N, H, W)
    gts = np.concatenate(gt_imgs)            # (N, H, W)
    pf, gf = probs.reshape(-1), gts.reshape(-1).astype(bool)
    sweep = []
    for t in thresholds:
        pred = pf >= t
        tp = int((pred & gf).sum()); fp = int((pred & ~gf).sum()); fn = int((~pred & gf).sum())
        obj = ObjectMetrics(overlap_thresh)
        for pimg, gimg in zip(probs, gts):
            obj.update((pimg >= t).astype(np.uint8), gimg)
        od = obj.compute()
        sweep.append({'threshold': round(float(t), 2),
                      'recall': round(tp / (tp + fn + 1e-9), 4),
                      'precision': round(tp / (tp + fp + 1e-9), 4),
                      'iou': round(tp / (tp + fp + fn + 1e-9), 4),
                      'detection_rate': round(od['detection_rate'], 4),
                      'missed': od['missed'], 'false_alarms': od['false_alarms']})
    ok = [r for r in sweep if r['recall'] >= target_recall]
    best = max(ok, key=lambda r: r['threshold']) if ok else max(sweep, key=lambda r: r['recall'])
    return {'recommended_threshold': best['threshold'], 'achieved_recall': best['recall'],
            'achieved_precision': best['precision'], 'sweep': sweep}

In [15]:
#| test
# test: threshold search returns a valid recommendation on a tiny val loader
from torch.utils.data import Dataset, DataLoader

class _Val(Dataset):
    def __len__(self): return 4
    def __getitem__(self, i):
        m = torch.zeros(16, 16, dtype=torch.long); m[4:12, 4:12] = 1
        return {'image': torch.rand(3, 16, 16), 'mask': m, 'sample_id': str(i)}

res = find_operating_threshold(mb, DataLoader(_Val(), batch_size=2), dev, target_recall=0.0, thresholds=[0.3, 0.5, 0.7])
assert 0.0 < res['recommended_threshold'] < 1.0 and len(res['sweep']) == 3

## Batched tile inference (large images)

`predict` runs one view at a time — fine for crops, but a full inspection image
(thousands of pixels per side) needs sliding-window tiling, and running **all** tiles in
a single forward pass OOMs while running them one-by-one leaves the GPU idle between
tile loads. `predict_tiled` closes that gap: tiles are forwarded in chunks of
`inference_batch_size` and the chunk outputs are stitched back with Gaussian-weighted
blending, with the same activation semantics as `predict` (sigmoid for binary/multilabel,
softmax for multiclass).

Plan (experiment → verify → export):

1. Tile-start math: clamped sliding window (last start shifted back so every tile fits),
   proven to cover every pixel including images smaller than one tile (padded up).
2. Salvage from the retired Oct-1 cluster (issue #27): the Gaussian blend weights move in
   here as a private helper (`_gaussian_weight_map`, was `stitching.gaussian_weight_map`),
   and `padding_mode='reflect'` re-implements the Oct-1 reflect option for images smaller
   than one tile — mirror-extend instead of zero-darkening real edge structure.
3. Batched forward + weighted accumulation: verify `inference_batch_size=1` gives
   **bit-identical** results to any larger value on a synthetic model — the knob trades
   memory against throughput only, never accuracy.
4. Promote to `predict_tiled`; edge cases: multilabel / multiclass heads, `activate=False`
   logits passthrough, `overlap >= tile_size`, `inference_batch_size < 1` and an unknown
   `padding_mode` fail clearly.

Issue #27 note: this is now the **single** tiled-inference interface — the parallel
Oct-1 cluster (`tiling_core` / `tiling` / `stitching` / `model_wrapper` / `onnx_export`)
is retired, Gaussian-weighted is the only merge strategy (alternatives deferred in the
spec, `docs/superpowers/specs/2026-10-03-deep-seam-consolidation.md`), and the zero-pad
default keeps today's behaviour bit-identical.

Production tie-in (issue #20): the batching knob is the `inference.inference_batch_size`
config surface, and a curated-encoder ONNX smoke covering the tiled FP32 path lives in
`nbs/13_export.ipynb`.

In [16]:
# experiment: tile-start math (clamped sliding window) — not exported yet
def _tile_starts_proto(extent, tile_size, overlap):
    if not (0 <= overlap < tile_size):
        raise ValueError(f'overlap must satisfy 0 <= overlap < tile_size ({tile_size}), got {overlap}')
    stride = tile_size - overlap
    if extent <= tile_size: return [0]
    n = -(-(extent - tile_size) // stride) + 1  # ceil((extent - tile_size) / stride) + 1 windows
    return sorted({min(i * stride, extent - tile_size) for i in range(n)})

for extent, ts, ov in [(256, 64, 16), (70, 64, 16), (64, 64, 16), (10, 64, 16), (130, 64, 0)]:
    starts = _tile_starts_proto(extent, ts, ov)
    padded = max(extent, ts)
    covered = torch.zeros(padded, dtype=torch.bool)
    for s in starts: covered[s:s + ts] = True
    assert covered.all() and starts[0] == 0, (extent, ts, ov, starts)
    print(f'extent={extent:4d} tile={ts} overlap={ov:2d} -> {len(starts)} starts {starts}')
print('tile math covers every pixel')

extent= 256 tile=64 overlap=16 -> 5 starts [0, 48, 96, 144, 192]
extent=  70 tile=64 overlap=16 -> 2 starts [0, 6]
extent=  64 tile=64 overlap=16 -> 1 starts [0]
extent=  10 tile=64 overlap=16 -> 1 starts [0]
extent= 130 tile=64 overlap= 0 -> 3 starts [0, 64, 66]
tile math covers every pixel


In [ ]:
# experiment: gaussian_weight_map moved in from the retired stitching module (private now)
def _gwm_proto(patch_size, sigma_factor=0.25, device=None):
    sigma = sigma_factor * patch_size
    coords_1d = torch.arange(patch_size, dtype=torch.float32, device=device)
    centre = (patch_size - 1) / 2.0
    gauss_1d = torch.exp(-0.5 * ((coords_1d - centre) / sigma) ** 2)
    w = torch.outer(gauss_1d, gauss_1d)
    return w / w.max()

w = _gwm_proto(64)
assert w.shape == (64, 64) and w.max() == 1.0 and w.min() > 0
assert w[32, 32] == 1.0 and w[0, 0] == w[0, 63] == w[63, 0] == w[63, 63]  # centre peak, symmetric corners
print('gaussian weight map verified')

In [ ]:
# experiment: reflect padding for images smaller than a tile — not exported yet
def _reflect_idx_proto(n, pad):
    if n == 1: return torch.zeros(n + pad, dtype=torch.long)  # a single pixel reflects to itself
    period = 2 * (n - 1)  # torch 'reflect' excludes the edge pixel, so the mirror repeats with this period
    m = torch.arange(n + pad) % period
    return torch.where(m < n, m, period - m)

def _reflect_pad_proto(x, ph, pw):
    idx_r, idx_c = _reflect_idx_proto(x.shape[-2], ph), _reflect_idx_proto(x.shape[-1], pw)
    return x[..., idx_r, :][..., :, idx_c]

import torch.nn.functional as F
x = torch.rand(1, 2, 7, 5)
assert torch.equal(_reflect_pad_proto(x, 3, 4), F.pad(x, (0, 4, 0, 3), mode='reflect')), 'F.pad parity'
big = _reflect_pad_proto(x, 20, 30)  # pad larger than the dim — F.pad raises, the index map does not
y = x; pl, cl = 20, 30
while pl > 0 or cl > 0:  # equivalence: grow in reflect rounds, what F.pad would do if allowed
    r, c = min(pl, y.shape[-2] - 1), min(cl, y.shape[-1] - 1)
    y = F.pad(y, (0, c, 0, r), mode='reflect'); pl, cl = pl - r, cl - c
assert torch.equal(big, y) and big.shape == (1, 2, 27, 35)
assert (_reflect_pad_proto(torch.ones(1, 1, 1, 4), 5, 0) == 1).all()  # 1-px dim -> self-replication
print('reflect pad verified: F.pad parity, oversized pad, 1-px dim')

In [17]:
# experiment: batched forward + Gaussian-weighted stitch on synthetic data
def _predict_tiled_proto(model, image, device, tile_size=64, overlap=16, inference_batch_size=2, task=None, activate=True, padding_mode='zero'):
    x = image.unsqueeze(0).to(device)
    H, W = x.shape[-2], x.shape[-1]
    ph, pw = max(0, tile_size - H), max(0, tile_size - W)
    if ph or pw:  # image smaller than a tile: zero (ONNX semantics) or mirror-extend
        x = _reflect_pad_proto(x, ph, pw) if padding_mode == 'reflect' \
            else torch.nn.functional.pad(x, (0, pw, 0, ph), mode='constant', value=0)
    Hp, Wp = x.shape[-2], x.shape[-1]
    coords = [(y, xx) for y in _tile_starts_proto(Hp, tile_size, overlap) for xx in _tile_starts_proto(Wp, tile_size, overlap)]
    tiles = torch.stack([x[0, :, y:y + tile_size, xx:xx + tile_size] for y, xx in coords])  # (N, C, ts, ts)
    def act(logits):
        if not activate: return logits
        return torch.sigmoid(logits) if (logits.shape[1] == 1 or task == 'multilabel') else torch.softmax(logits, 1)
    wmap = _gwm_proto(tile_size, device=x.device)
    acc, wsum = None, x.new_zeros(1, Hp, Wp)
    for i in range(0, len(coords), inference_batch_size):  # <- the batching knob
        probs = act(model(tiles[i:i + inference_batch_size]))
        if acc is None: acc = probs.new_zeros(probs.shape[1], Hp, Wp)  # sized by OUTPUT channels
        for j, (y, xx) in enumerate(coords[i:i + inference_batch_size]):
            acc[:, y:y + tile_size, xx:xx + tile_size] += wmap * probs[j]
            wsum[:, y:y + tile_size, xx:xx + tile_size] += wmap
    return (acc / wsum)[:, :H, :W]  # (C_out, H, W)

import torch.nn as nn
mb = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Sigmoid())
img = torch.rand(1, 40, 50)
p1  = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=1)
p2  = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=2)
p99 = _predict_tiled_proto(mb, img, 'cpu', inference_batch_size=999)
assert p2.shape == (1, 40, 50)
assert torch.equal(p1, p2) and torch.equal(p2, p99), 'batch-size invariance failed'
assert 0 <= p2.min() and p2.max() <= 1
# verify: image smaller than a tile; multilabel head; logits passthrough
assert _predict_tiled_proto(mb, torch.rand(1, 10, 12), 'cpu', tile_size=64).shape == (1, 10, 12)
pm = _predict_tiled_proto(nn.Conv2d(1, 3, 3, padding=1), img, 'cpu', task='multilabel')
assert pm.shape == (3, 40, 50) and 0 <= pm.min() and pm.max() <= 1
lo = _predict_tiled_proto(nn.Conv2d(1, 1, 3, padding=1), img, 'cpu', activate=False)
assert bool(lo.min() < 0 or lo.max() > 1)
# verify: reflect differs on a small edge-structure image; modes agree when no padding is needed
edge_img = torch.zeros(1, 20, 24); edge_img[:, :6] = 1.0  # bright band at the top edge
pz = _predict_tiled_proto(mb, edge_img, 'cpu', tile_size=32)
pr = _predict_tiled_proto(mb, edge_img, 'cpu', tile_size=32, overlap=8, padding_mode='reflect')
assert not torch.allclose(pz, pr), 'reflect should differ from zero on edge structure'
big = torch.rand(1, 128, 160)  # larger than a tile in both dims -> no padding -> identical
assert torch.equal(_predict_tiled_proto(mb, big, 'cpu', tile_size=64),
                   _predict_tiled_proto(mb, big, 'cpu', tile_size=64, padding_mode='reflect')), 'modes must agree without padding'
print('prototype verified: batch-invariant, padded, reflect option, multilabel, logits passthrough')

prototype verified: batch-invariant, padded, multilabel, logits passthrough


In [18]:
#| export
def _reflect_idx(
    n: int,   # size of the original axis
    pad: int, # how much to extend the axis
) -> torch.Tensor:
    """Mirror indices into an axis of size `n` extended by `pad` (torch 'reflect' semantics).

    Unlike `torch.nn.functional.pad(..., mode='reflect')` this works for any `pad`,
    including larger than the axis itself, and degrades to self-replication on a 1-px axis.

    Example:
        >>> _reflect_idx(4, 3)
        tensor([0, 1, 2, 3, 2, 1, 0])
    """
    if n == 1: return torch.zeros(n + pad, dtype=torch.long)  # a single pixel reflects to itself
    period = 2 * (n - 1)  # torch 'reflect' excludes the edge pixel, so the mirror repeats with this period
    m = torch.arange(n + pad) % period
    return torch.where(m < n, m, period - m)

def _reflect_pad(
    x: torch.Tensor, # (B, C, H, W) tensor
    ph: int,         # rows to add at the bottom
    pw: int,         # columns to add at the right
) -> torch.Tensor:
    """Bottom/right reflect padding that works for any `ph`, `pw` (see `_reflect_idx`)."""
    if not (ph or pw): return x
    idx_r, idx_c = _reflect_idx(x.shape[-2], ph), _reflect_idx(x.shape[-1], pw)
    return x[..., idx_r, :][..., :, idx_c]

def _gaussian_weight_map(
    patch_size: int,             # tile side (assumed square)
    sigma_factor: float = 0.25,  # smaller -> sharper centre weighting
    device: torch.device = None, # target device for the returned tensor
) -> torch.Tensor:
    """Build a 2-D Gaussian weight map `(patch_size, patch_size)` with max weight 1.0.

    Example:
        >>> w = _gaussian_weight_map(64); (w.shape, w.max())
        (torch.Size([64, 64]), tensor(1.))
    """
    sigma = sigma_factor * patch_size
    coords_1d = torch.arange(patch_size, dtype=torch.float32, device=device)
    centre = (patch_size - 1) / 2.0
    gauss_1d = torch.exp(-0.5 * ((coords_1d - centre) / sigma) ** 2)
    weight_map = torch.outer(gauss_1d, gauss_1d)
    return weight_map / weight_map.max()  # normalise to [0, 1]

def tile_starts(
    extent: int,     # image extent along one axis (H or W), already padded up if smaller than tile_size
    tile_size: int,  # tile side along the same axis
    overlap: int,    # overlap between adjacent tiles along the same axis
) -> List[int]:
    """Clamped sliding-window starts covering `[0, extent)`: 0 <= overlap < tile_size required.

    The last start is shifted back so every tile fits inside the extent — no output pixels
    are left uncovered and no tile runs past the edge.

    Example:
        >>> tile_starts(256, 64, 16)
        [0, 48, 96, 144, 192]
    """
    if not (0 <= overlap < tile_size):
        raise ValueError(f'overlap must satisfy 0 <= overlap < tile_size ({tile_size}), got {overlap}')
    stride = tile_size - overlap
    if extent <= tile_size: return [0]
    n = -(-(extent - tile_size) // stride) + 1  # ceil((extent - tile_size) / stride) + 1 windows
    return sorted({min(i * stride, extent - tile_size) for i in range(n)})

@torch.no_grad()
def predict_tiled(
    model,                          # trained model returning (B, C, H, W) logits
    image: torch.Tensor,            # a single (C, H, W) input — any H, W (smaller images are padded up)
    device: torch.device,           # cpu or cuda
    tile_size: int = 256,           # tile side fed to the model
    overlap: int = 32,              # adjacent-tile overlap; Gaussian blending stitches tiles back
    inference_batch_size: int = 8,  # tiles per forward — raise for throughput, lower if OOM
    activate: bool = True,          # apply sigmoid/softmax to logits (same as `predict`)
    task: str = None,               # 'binary' | 'multiclass' | 'multilabel'; None infers like `predict`
    padding_mode: str = 'zero',     # 'zero' (default, ONNX-matching) or 'reflect' for images smaller than a tile
) -> torch.Tensor:
    """Return `(C, H, W)` probabilities for a large image via batched sliding-window tiles.

    Tiles are forwarded in chunks of `inference_batch_size` (the `inference.inference_batch_size`
    config knob) and stitched with Gaussian-weighted blending. `inference_batch_size=1` gives
    bit-identical output to any larger value — the knob trades GPU memory against throughput
    only, never accuracy. Activation semantics match `predict`. `padding_mode` only matters when
    the image is smaller than one tile: 'zero' (default) keeps ONNX-export semantics, 'reflect'
    mirror-extends the edge so real structure near the border is not darkened.

    Example:
        >>> prob = predict_tiled(model, torch.rand(1, 1152, 1632), dev, tile_size=256, overlap=32)
        >>> mask = (prob[0] >= 0.5).long()
    """
    if inference_batch_size < 1:
        raise ValueError(f'inference_batch_size must be >= 1, got {inference_batch_size}')
    if padding_mode not in ('zero', 'reflect'):
        raise ValueError(f"padding_mode must be 'zero' or 'reflect', got {padding_mode!r}")
    model.eval()
    x = image.unsqueeze(0).to(device)
    H, W = x.shape[-2], x.shape[-1]
    ph, pw = max(0, tile_size - H), max(0, tile_size - W)
    if ph or pw:  # image smaller than a tile
        x = _reflect_pad(x, ph, pw) if padding_mode == 'reflect' \
            else torch.nn.functional.pad(x, (0, pw, 0, ph), mode='constant', value=0)
    Hp, Wp = x.shape[-2], x.shape[-1]
    coords = [(y, xx) for y in tile_starts(Hp, tile_size, overlap)
              for xx in tile_starts(Wp, tile_size, overlap)]  # row-major, fixed order
    tiles = torch.stack([x[0, :, y:y + tile_size, xx:xx + tile_size] for y, xx in coords])  # (N, C, ts, ts)

    wmap = _gaussian_weight_map(tile_size, device=x.device)
    acc, wsum = None, x.new_zeros(1, Hp, Wp)
    for i in range(0, len(coords), inference_batch_size):
        probs = _activate(model(tiles[i:i + inference_batch_size]), task, activate)
        if acc is None: acc = probs.new_zeros(probs.shape[1], Hp, Wp)  # output-channel aware
        for j, (y, xx) in enumerate(coords[i:i + inference_batch_size]):
            acc[:, y:y + tile_size, xx:xx + tile_size] += wmap * probs[j]
            wsum[:, y:y + tile_size, xx:xx + tile_size] += wmap
    return (acc / wsum)[:, :H, :W]  # (C_out, H, W)

In [19]:
#| test
# test: predict_tiled on synthetic data (same checks as the experiment cells, via exports)
import torch.nn as nn

# zero-pad parity vs pre-refactor (issue #27 regression guard, fixed seed)
torch.manual_seed(0)
mb_ref = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Sigmoid())
p_ref = predict_tiled(mb_ref, torch.rand(1, 40, 50), dev, tile_size=32, overlap=8, inference_batch_size=4)
assert abs(p_ref.double().sum().item() - 1246.5318177342415) < 1e-6

mb = nn.Sequential(nn.Conv2d(1, 1, 3, padding=1), nn.Sigmoid())
img = torch.rand(1, 40, 50)
p1  = predict_tiled(mb, img, dev, inference_batch_size=1)
p2  = predict_tiled(mb, img, dev, inference_batch_size=2)
p99 = predict_tiled(mb, img, dev, inference_batch_size=999)
assert p2.shape == (1, 40, 50)
assert torch.equal(p1, p2) and torch.equal(p2, p99), 'batch-size invariance failed'
assert 0 <= p2.min() and p2.max() <= 1
assert tile_starts(256, 64, 16) == [0, 48, 96, 144, 192]
# image smaller than one tile; multichannel input; multiclass / multilabel heads; logits passthrough
assert predict_tiled(mb, torch.rand(1, 10, 12), dev, tile_size=64).shape == (1, 10, 12)
assert predict_tiled(nn.Conv2d(3, 1, 3, padding=1), torch.rand(3, 40, 50), dev).shape == (1, 40, 50)
assert predict_tiled(nn.Conv2d(1, 4, 3, padding=1), img, dev).shape == (4, 40, 50)
pm = predict_tiled(nn.Conv2d(1, 3, 3, padding=1), img, dev, task='multilabel')
assert pm.shape == (3, 40, 50) and 0 <= pm.min() and pm.max() <= 1
lo = predict_tiled(nn.Conv2d(1, 1, 3, padding=1), img, dev, activate=False)
assert bool(lo.min() < 0 or lo.max() > 1)
# reflect: differs on a small edge-structure image; modes agree when no padding is needed
edge_img = torch.zeros(1, 20, 24); edge_img[:, :6] = 1.0  # bright band at the top edge
pz = predict_tiled(mb, edge_img, dev, tile_size=32, overlap=8)
pr = predict_tiled(mb, edge_img, dev, tile_size=32, overlap=8, padding_mode='reflect')
assert not torch.allclose(pz, pr), 'reflect should differ from zero on edge structure'
big = torch.rand(1, 128, 160)  # larger than a tile in both dims -> no padding -> identical
assert torch.equal(predict_tiled(mb, big, dev, tile_size=64),
                   predict_tiled(mb, big, dev, tile_size=64, padding_mode='reflect'))
# clear failures on bad knobs
for bad in ({'overlap': 256}, {'inference_batch_size': 0}, {'padding_mode': 'mirror'}):
    try:
        predict_tiled(mb, img, dev, **bad); raise AssertionError(f'no error for {bad}')
    except ValueError:
        pass

## Instance detections to a semantic map

An instance model such as RF-DETR segmentation (research: `docs/superpowers/specs/2026-11-22-rfdetr-seg-integration-research.md`) returns objects, not a per-pixel map: for each detection a boolean mask `(H, W)`, a class id and a confidence score. `SegMetrics`, `ObjectMetrics` and `review_report.Sample` all want a semantic map, so we paint the detections onto one. This function takes plain arrays; it never imports `rfdetr` or `supervision` (a `supervision.Detections` works because it has the same three attributes: `.mask (N, H, W) bool`, `.class_id (N,)`, `.confidence (N,)`).

Plan:

1. Drop detections below `threshold` and classes not in `class_ids`.
2. Paint the detections from lowest to highest score, so that where two overlap **the higher score wins**.
3. `output='labels'` gives `(H, W)` long (0 = background, otherwise the detection's `class_id`); `output='stack'` gives `(K, H, W)` float, one channel per entry of `class_ids`, holding the best score covering each pixel (0 where nothing).
4. No detections gives an all-background map of the requested `shape`.
5. Class ids start at `first_class_id` (default 1; RF-DETR's 2-class model returned 0-based ids, so it needs `first_class_id=0`); anything below raises instead of painting background. Equal scores: the later detection wins, deterministically.

In [ ]:
# experiment: two overlapping instances of class 1 and one of class 2, scores decide the overlap
H, W = 12, 12
masks_e = np.zeros((3, H, W), bool)
masks_e[0, 2:8, 2:8] = True      # score 0.6
masks_e[1, 5:11, 5:11] = True    # score 0.9, overlaps the first on 5:8
masks_e[2, 0:2, 9:12] = True     # class 2, score 0.8
cls_e = np.array([1, 1, 2]); conf_e = np.array([0.6, 0.9, 0.8])
lab = np.zeros((H, W), np.int64)
for k in np.argsort(conf_e):                     # lowest first, so the highest score paints last
    lab[masks_e[k]] = cls_e[k]
print(lab)
assert (lab[5:8, 5:8] == 1).all() and lab[0, 10] == 2 and lab[2, 2] == 1
best = np.zeros((2, H, W), np.float32)           # stack: channel per class, max score per pixel
for k in range(3):
    c = int(cls_e[k]) - 1
    best[c] = np.where(masks_e[k], np.maximum(best[c], conf_e[k]), best[c])
assert best[0, 6, 6] == np.float32(0.9) and best[0, 2, 2] == np.float32(0.6) and best[1, 0, 10] == np.float32(0.8)

In [ ]:
#| export
def _class_id_hint(what: str, bad, first_class_id: int) -> str:
    "Error text for a class id below `first_class_id`, with the fix for models whose ids start at 0."
    hint = ' A model with 0-based class ids (RF-DETR) needs first_class_id=0.' if first_class_id == 1 and 0 in bad else ''
    return f'class ids start at {first_class_id}; got {what} {bad}.{hint}'

def _unpack_dets(
    dets,                    # `.mask/.class_id/.confidence` object or `(mask, class_id, confidence)` tuple
    shape: Optional[tuple],  # (H, W) or None
    first_class_id: int = 1, # smallest valid class id
) -> tuple:
    "Read, coerce and validate detections: `(N, H, W)` bool, `(N,)` int64, `(N,)` float32. Class ids must be >= `first_class_id`."
    if isinstance(dets, (tuple, list)):
        mask, cid, conf = dets
    else:
        mask, cid, conf = dets.mask, dets.class_id, dets.confidence
    if mask is None or np.asarray(mask).ndim != 3:
        if shape is None:
            raise ValueError('shape is required when there is no (N, H, W) mask')
        mask, cid, conf = np.zeros((0,) + tuple(shape), bool), np.zeros(0, np.int64), np.zeros(0, np.float32)
    mask, cid, conf = np.asarray(mask).astype(bool), np.asarray(cid).astype(np.int64), np.asarray(conf, np.float32)
    if mask.ndim != 3 or not (len(mask) == len(cid) == len(conf)):
        raise ValueError(f'mask (N,H,W), class_id (N,), confidence (N,) must agree; got {mask.shape}, {cid.shape}, {conf.shape}')
    if shape is not None and tuple(mask.shape[1:]) != tuple(shape):
        raise ValueError(f'mask shape {mask.shape[1:]} != requested shape {tuple(shape)}')
    if (cid < first_class_id).any():
        raise ValueError(_class_id_hint('class_id', sorted(set(cid[cid < first_class_id].tolist())), first_class_id))
    return mask, cid, conf

def detections_to_semantic(
    dets,                                # object with `.mask (N,H,W) bool`, `.class_id (N,)`, `.confidence (N,)`, or a `(mask, class_id, confidence)` tuple
    shape: Optional[tuple] = None,       # (H, W); needed only when there are no detections to read it from
    class_ids: Optional[Sequence[int]] = None,  # classes to keep, in channel order for `output='stack'`; None = every class present
    rule: str = 'score',                 # overlap rule: 'score' = the higher confidence wins
    output: str = 'labels',              # 'labels' -> (H, W) long | 'stack' -> (K, H, W) float32 best score per class
    threshold: float = 0.0,              # drop detections with confidence below this
    first_class_id: int = 1,             # the model's smallest class id: 1 (default) or 0 for 0-based ids such as RF-DETR's
):
    """Paint instance detections onto a semantic map so `SegMetrics`, `ObjectMetrics` and `review_report.Sample` can consume them.

    Pure numpy in and out; no `rfdetr` / `supervision` import. Labels hold `class_id - first_class_id + 1`, so 0 stays background.

    Class ids must be >= `first_class_id`: a detection or an entry of `class_ids` below it raises `ValueError` instead
    of silently painting background. RF-DETR returns **0-based** ids (indices into `model.class_names`, checked on a
    2-class model), so pass `first_class_id=0` for it and `class_ids=[0, 1, ...]` in the model's own numbering. Where two detections overlap the higher score wins; with
    **equal scores the later detection (higher index) wins**, so the result is deterministic. `output='stack'` keeps
    the best score per class and pixel, so ties do not matter there.

    Example:
        >>> m = np.zeros((1, 4, 4), bool); m[0, :2, :2] = True
        >>> detections_to_semantic((m, np.array([1]), np.array([0.9])), class_ids=[1]).sum()
        4
    """
    if rule != 'score':
        raise ValueError(f"rule must be 'score', got {rule!r}")
    if output not in ('labels', 'stack'):
        raise ValueError(f"output must be 'labels' or 'stack', got {output!r}")
    mask, cid, conf = _unpack_dets(dets, shape, first_class_id)
    h, w = mask.shape[1:]
    keep_cls = np.unique(cid) if class_ids is None else np.asarray(list(class_ids), np.int64)
    if (keep_cls < first_class_id).any():
        raise ValueError(_class_id_hint('class_ids', keep_cls.tolist(), first_class_id))
    keep = (conf >= threshold) & np.isin(cid, keep_cls)
    # stable sort keeps detection order among equal scores, so the later one paints last and wins
    order = [k for k in np.argsort(conf, kind='stable') if keep[k]]
    if output == 'labels':
        out = np.zeros((h, w), np.int64)
        for k in order:
            out[mask[k]] = cid[k] - first_class_id + 1
        return out
    out = np.zeros((len(keep_cls), h, w), np.float32)
    channel_of = {int(c): i for i, c in enumerate(keep_cls)}
    for k in order:
        channel = out[channel_of[int(cid[k])]]
        channel[mask[k]] = np.maximum(channel[mask[k]], conf[k])
    return out

In [ ]:
#| test
# test: the experiment's expectations, plus layouts, filters, empty input and the consumers
from types import SimpleNamespace
d = SimpleNamespace(mask=masks_e, class_id=cls_e, confidence=conf_e)
lab = detections_to_semantic(d)
assert lab.dtype == np.int64 and lab.shape == (12, 12)
assert (lab[5:8, 5:8] == 1).all() and lab[0, 10] == 2 and lab[2, 2] == 1 and lab[11, 0] == 0
# higher score wins regardless of detection order
perm = [2, 1, 0]
d2 = SimpleNamespace(mask=masks_e[perm], class_id=cls_e[perm], confidence=conf_e[perm])
assert (detections_to_semantic(d2) == lab).all()
# a class where the lower score is the *other* class: score, not class id, decides
m2 = np.zeros((2, 4, 4), bool); m2[:, :] = True
assert (detections_to_semantic((m2, np.array([2, 1]), np.array([0.3, 0.7]))) == 1).all()
assert (detections_to_semantic((m2, np.array([2, 1]), np.array([0.9, 0.7]))) == 2).all()
st = detections_to_semantic(d, class_ids=[1, 2], output='stack')
assert st.shape == (2, 12, 12) and st.dtype == np.float32
assert np.isclose(st[0, 6, 6], 0.9) and np.isclose(st[0, 2, 2], 0.6) and np.isclose(st[1, 0, 10], 0.8) and st[1, 6, 6] == 0
# class filter and threshold
assert (detections_to_semantic(d, class_ids=[2]) > 0).sum() == 6 and detections_to_semantic(d, class_ids=[2]).max() == 2
assert detections_to_semantic(d, threshold=0.7)[2, 2] == 0 and detections_to_semantic(d, threshold=0.7)[6, 6] == 1
# no detections: background map of the requested shape; shape mismatch and bad args raise
e = SimpleNamespace(mask=np.zeros((0, 12, 12), bool), class_id=np.zeros(0, int), confidence=np.zeros(0))
assert detections_to_semantic(e).sum() == 0 and detections_to_semantic(e, output='stack', class_ids=[1]).shape == (1, 12, 12)
assert detections_to_semantic(SimpleNamespace(mask=None, class_id=None, confidence=None), shape=(5, 7)).shape == (5, 7)
for bad in (dict(rule='area'), dict(output='x'), dict(shape=(3, 3))):
    try: detections_to_semantic(d, **bad); raise AssertionError(bad)
    except ValueError: pass
# consumers: ObjectMetrics on a binary view, SegMetrics on stack scores turned into logits
gt = np.zeros((12, 12), np.uint8); gt[2:8, 2:8] = 1; gt[5:11, 5:11] = 1
om = ObjectMetrics(); om.update((lab == 1).astype(np.uint8), gt)
assert om.compute()['detection_rate'] > 0.99
from segmentation_tools_pt.metrics import SegMetrics
sm = SegMetrics(num_classes=1)
sc = torch.from_numpy(st[:1]).clamp(1e-4, 1 - 1e-4)
sm.update(torch.logit(sc)[None], torch.from_numpy(gt.astype(np.float32))[None])
assert sm.compute()['iou'] > 0.99

In [ ]:
#| test
# test: equal scores are deterministic (the later detection wins); class id 0 and below is refused
tie = np.ones((2, 4, 4), bool)
assert (detections_to_semantic((tie, np.array([1, 2]), np.array([0.5, 0.5]))) == 2).all()
assert (detections_to_semantic((tie, np.array([2, 1]), np.array([0.5, 0.5]))) == 1).all()
for ids in ([0, 1], [1, -1]):
    try: detections_to_semantic((tie, np.array(ids), np.array([0.5, 0.5]))); raise AssertionError(ids)
    except ValueError as e: assert 'class' in str(e)
try: detections_to_semantic((tie, np.array([1, 2]), np.array([0.5, 0.5])), class_ids=[0, 1]); raise AssertionError
except ValueError as e: assert 'class' in str(e)

In [ ]:
#| test
# test: models with 0-based class ids (RF-DETR returns indices into `class_names`): first_class_id=0 shifts them to labels 1..K
m3 = np.zeros((2, 4, 4), bool); m3[0, :2] = True; m3[1, 2:] = True
d3 = (m3, np.array([0, 1]), np.array([0.9, 0.8]))
lab0 = detections_to_semantic(d3, first_class_id=0)
assert lab0.dtype == np.int64 and (lab0[:2] == 1).all() and (lab0[2:] == 2).all()   # ids 0, 1 -> labels 1, 2; background stays 0
st0 = detections_to_semantic(d3, class_ids=[0, 1], output='stack', first_class_id=0)
assert st0.shape == (2, 4, 4) and np.isclose(st0[0, 0, 0], 0.9) and st0[0, 3, 0] == 0 and np.isclose(st0[1, 3, 0], 0.8)
assert (detections_to_semantic(d3, class_ids=[1], first_class_id=0) == np.where(m3[1], 2, 0)).all()   # class filter uses the model's ids
# the default (first_class_id=1) still refuses id 0 and says how to fix it; ids below first_class_id refuse too
try: detections_to_semantic(d3); raise AssertionError
except ValueError as e: assert 'first_class_id=0' in str(e), e
try: detections_to_semantic((m3, np.array([-1, 1]), np.array([0.9, 0.8])), first_class_id=0); raise AssertionError
except ValueError as e: assert 'class' in str(e)

In [20]:
#| hide
import nbdev; nbdev.nbdev_export()